# sensor_select_wells

**Purpose:** Pick the wells for the PI-style simulated sensors (DEC-032): wells with OIL or GAS volume in the
configured month, spread over several reporting facilities. Read-only on the Lakehouse tables.

**Input:** `Files/config/sensor.json` (`month`, `selection`); tables `gold_fact_production`, `gold_dim_well`.

**Output:** `Files/sensor/well_selection.json`: one entry per well with its facility, product and the real Petrinex
volume and hours of the month. Copy the `wells` list into `config/sensor.json`; the generator uses that frozen
snapshot, and `05_sensor_gold` reconciles against `gold_fact_production` itself.

**Selection:** per well, the month's OIL and GAS volume, maximum hours and number of reporting facilities. A well
qualifies with one reporting facility, hours >= `min_hours` and a primary product (OIL if oil volume >=
`min_volume.OIL`, else GAS if gas volume >= `min_volume.GAS`). For each product, facilities with at least
`wells_per_facility` qualifying wells are ranked by a seeded hash and the first `facilities_per_product` are taken;
inside each facility, wells are ranked by the same hash. Placeholder wells (not in ST37) are excluded.

Run on its own, not in a pipeline: only when the selection settings change.


In [ ]:
import json
from pyspark.sql import functions as F
from pyspark.sql.window import Window

with open("/lakehouse/default/Files/config/sensor.json") as fh:
    CFG = json.load(fh)
SEL = CFG["selection"]
MONTH = CFG["month"]["production_month"] + "-01"

fact = spark.table("gold_fact_production").filter(F.col("production_month") == MONTH)
real_wells = spark.table("gold_dim_well").filter(~F.coalesce(F.col("is_placeholder"), F.lit(False))).select("well_key")

per_well = (fact.filter(F.col("ProductID").isin(SEL["products"]))
    .groupBy("well_key")
    .agg(F.sum(F.when(F.col("ProductID") == "OIL", F.col("volume"))).alias("oil_volume"),
         F.sum(F.when(F.col("ProductID") == "GAS", F.col("volume"))).alias("gas_volume"),
         F.max("hours").alias("hours"),
         F.countDistinct("ReportingFacilityID").alias("n_facilities"),
         F.first("ReportingFacilityID").alias("facility_id"))
    .join(real_wells, "well_key", "left_semi")
    .filter((F.col("n_facilities") == 1) & (F.col("hours") >= SEL["min_hours"]))
    .withColumn("product",
        F.when(F.col("oil_volume") >= SEL["min_volume"]["OIL"], "OIL")
         .when(F.col("gas_volume") >= SEL["min_volume"]["GAS"], "GAS"))
    .filter(F.col("product").isNotNull())
    .withColumn("volume", F.when(F.col("product") == "OIL", F.col("oil_volume")).otherwise(F.col("gas_volume"))))

seed = F.lit(str(SEL["seed"]))
eligible = (per_well.groupBy("product", "facility_id").count()
    .filter(F.col("count") >= SEL["wells_per_facility"])
    .withColumn("rank", F.row_number().over(Window.partitionBy("product")
                                            .orderBy(F.xxhash64(seed, F.col("facility_id")), "facility_id")))
    .filter(F.col("rank") <= SEL["facilities_per_product"])
    .select("product", "facility_id"))

chosen = (per_well.join(eligible, ["product", "facility_id"])
    .withColumn("rank", F.row_number().over(Window.partitionBy("product", "facility_id")
                                            .orderBy(F.xxhash64(seed, F.col("well_key")), "well_key")))
    .filter(F.col("rank") <= SEL["wells_per_facility"])
    .orderBy("product", "facility_id", "well_key")
    .select("well_key", "facility_id", "product",
            F.col("volume").cast("double").alias("petrinex_volume"), F.col("hours").cast("int").alias("petrinex_hours")))

rows = [r.asDict() for r in chosen.collect()]
expected = 2 * SEL["facilities_per_product"] * SEL["wells_per_facility"]
if len(rows) != expected:
    raise ValueError(f"selected {len(rows)} wells, expected {expected}: relax the selection settings")
run_id = spark.table("gold_fact_production").select("_gold_run_id").first()[0]
out = {"source": {"table": "gold_fact_production", "gold_run_id": run_id, "production_month": CFG["month"]["production_month"]},
       "wells": rows}
notebookutils.fs.put("Files/sensor/well_selection.json", json.dumps(out, indent=1), True)
print(len(rows), "wells from", len({r["facility_id"] for r in rows}), "facilities; gold run", run_id)
display(chosen)
